In [29]:

import random
from dataclasses import dataclass


# Juegos

En esta práctica programaremos agentes capaces de jugar. Los juegos que trataremos tendrán estas características:

* Suma cero.
* Dos jugadores que se turnan.
* Información perfecta.
* Transiciones deterministas.
* Horizonte finito (por tanto, no hay ciclos).

## Tres en raya

El tres en raya es un juego sencillo que nos permitirá introducir los conceptos básicos de la programación de juegos. El tablero tiene 3 × 3 casillas. Cada jugador dispone de una ficha que puede colocar en una casilla vacía. Gana quien consiga colocar tres fichas en línea (horizontal, vertical o diagonal). Si se llena el tablero sin que nadie lo consiga, la partida termina en empate.

### Representación de los estados

1. Para representar los estados del juego, implementa una clase de datos llamada `EstatJoc` con estos cuatro atributos:

* `jugador`: jugador al que le toca realizar el siguiente movimiento.
* `utilitat`: utilidad del estado para el jugador.
* `tauler`: representación del tablero.
* `moviments`: movimientos posibles desde el estado actual.

In [30]:
@dataclass(frozen=True)
class EstatJoc:
    torn: str
    utilitat: int
    tauler: dict
    moviments: list

### Clase `TresEnRatlla`

Nuestros juegos heredarán de la clase `Joc`, que implementa los métodos básicos para programar juegos. Estos son sus métodos:

* `accions(self, estat)`: devuelve los movimientos posibles desde el estado recibido como parámetro.
* `resultat(self, estat, accio)`: devuelve el estado resultante de aplicar la acción.
* `es_terminal(self, estat)`: devuelve `True` si el estado es terminal.
* `utilitat(self, estat, jugador)`: devuelve la utilidad del estado para el jugador.

In [31]:
class Joc:
    """Un joc és similar a un problema, però té una utilitat per a cada estat i un test terminal en lloc d'un cost de ruta i un test de l'objectiu. Per crear un joc, subclasseu aquesta classe i implementeu accions, resultats, utilitat, i test_terminal. Podeu sobreescriure la visualització i els successors o bé podeu heretar els seus mètodes per defecte. També necessitareu establir l'atribut .inicial a l'estat inicial; això es pot fer en el constructor."""

    def accions(self, estat):
        """Retorna una llista dels moviments permesos en aquest punt."""
        raise NotImplementedError

    def resultat(self, estat, moviment):
        """Retorna l'estat que resulta de fer un moviment des d'un estat."""
        raise NotImplementedError

    def utilitat(self, estat, jugador):
        """Retorna el valor d'aquest estat final per al jugador."""
        raise NotImplementedError

    def test_terminal(self, estat):
        """Retorna True si aquest és un estat final per al joc."""
        return not self.accions(estat)

    def torn(self, estat):
        """Retorna el jugador al qual li toca moure en aquest estat."""
        return estat.torn

    def display(self, estat):
        """Imprimeix o mostra d'alguna manera l'estat."""
        print(estat)

    def __repr__(self):
        return '<{}>'.format(self.__class__.__name__)

    def juga_joc(self, *jugadors):
        """Juga un joc de n-persones, alternant torns."""
        estat = self.inicial
        while True:
            for jugador in jugadors:
                moviment = jugador(self, estat)
                estat = self.resultat(estat, moviment)
                self.display(estat)
                if self.test_terminal(estat):
                    self.display(estat)
                    print()
                    return self.utilitat(estat, self.torn(self.inicial))

2. Crea una clase llamada `TresEnRatlla` que herede de `Joc`. Ten en cuenta estas condiciones:

* Los jugadores se representarán con las cadenas `'X'` y `'O'`.
* Los movimientos se representarán con tuplas de dos elementos: las coordenadas de la casilla donde se coloca la ficha.
* El tablero se representará con un diccionario cuyas claves serán las coordenadas y cuyo valor será el jugador que ocupa cada casilla.
* La utilidad será 1 si gana `'X'`, -1 si gana `'O'` y 0 si hay empate.
* El estado inicial tendrá como jugador a `'X'`, el tablero vacío y todas las casillas disponibles.
* Para facilitar la implementación de juegos mayores, el constructor tendrá tres parámetros opcionales: `h`, `v` y `k`. `h` y `v` son las dimensiones del tablero, y `k` es el número de fichas alineadas necesario para ganar. Por defecto, `h=3`, `v=3` y `k=3`.

In [32]:
class TresEnRatlla(Joc):
    """Joc del tres en ratlla (generalitzat a k) en un tauler h*v on guanya el primer que aconsegueix k en ratlla.
    Un estat es representa com una tupla (torn, tauler, moviments) on:
        - torn és el jugador a moure ('X' o 'O')
        - utilitat és la utilitat d'eixe estat, precalculada per eficiència.
        - moviments és la llista de caselles buides del tauler
        - tauler és un diccionari que conté per cada casella (x, y) del tauler el jugador que té la fitxa en eixa casella
        - moviments és la llista de caselles buides del tauler
    """

    def __init__(self, h=3, v=3, k=3):
        self.h = h
        self.v = v
        self.k = k
        moviments = [(x, y) for x in range(1, h + 1)
                     for y in range(1, v + 1)]
        self.inicial = EstatJoc(
            torn='X', utilitat=0, tauler={}, moviments=moviments
        )

    def accions(self, estat):
        """Les accions legals en un estat són les caselles buides."""
        return estat.moviments

    def resultat(self, estat, moviment):
        """Retorna l'estat que resulta d'aplicar un moviment a un estat."""
        if moviment not in estat.moviments:
            return estat  # El moviment il·legal no té cap efecte
        tauler = estat.tauler.copy()
        tauler[moviment] = estat.torn
        moviments = list(estat.moviments)
        moviments.remove(moviment)
        return EstatJoc(
            torn=('O' if estat.torn == 'X' else 'X'),
            utilitat=self.calcula_utilitat(
                tauler, moviment, estat.torn
            ),
            tauler=tauler, moviments=moviments
        )

    def utilitat(self, estat, jugador):
        """Retorna la utilitat d'un estat per a un jugador. El valor és 1 si el jugador guanya; -1 si el contrari guanya; 0 en cas d'empat."""

        return estat.utilitat if jugador == 'X' else -estat.utilitat

    def test_terminal(self, estat):
        """Un estat és terminal si algun jugador ha guanyat o no queden moviments."""
        return estat.utilitat != 0 or len(estat.moviments) == 0

    def display(self, estat):
        """Mostra l'estat actual del joc."""
        tauler = estat.tauler
        for x in range(1, self.h + 1):
            for y in range(1, self.v + 1):
                print(tauler.get((x, y), '.'), end=' ')
            print()

    def calcula_utilitat(self, tauler, estat, jugador):
        """Si 'X' guanya amb aquest moviment, retorna 1; si 'O' guanya retorna -1; si no retorna 0."""
        if (self.k_en_ralla(tauler, estat, jugador, (0, 1)) or
                self.k_en_ralla(tauler, estat, jugador, (1, 0)) or
                self.k_en_ralla(tauler, estat, jugador, (1, -1)) or
                self.k_en_ralla(tauler, estat, jugador, (1, 1))):
            return +1 if jugador == 'X' else -1
        else:
            return 0

    def k_en_ralla(self, tauler, moviment, jugador, delta_x_y, new_k=None):
        """Retorna veritat si hi ha una línia a través del moviment al tauler per al jugador.
        
        """
        (delta_x, delta_y) = delta_x_y
        x, y = moviment
        n = 0  # n és el nombre de moviments en línia
        while tauler.get((x, y)) == jugador:
            n += 1
            x, y = x + delta_x, y + delta_y
        x, y = moviment
        while tauler.get((x, y)) == jugador:
            n += 1
            x, y = x - delta_x, y - delta_y
        n -= 1  # Perquè hem comptat dos cops el mateix moviment
        return n >= self.k

### Función `jugador_aleatori`

Para jugar al tres en raya necesitaremos un jugador aleatorio, que elegirá uno de los movimientos posibles al azar.

3. Implementa la función `jugador_aleatori`, que recibe un juego y un estado y devuelve un movimiento aleatorio entre los posibles.

In [33]:
def jugador_aleatori(joc, estat):
    return random.choice(joc.accions(estat))

4. Comprueba que el jugador aleatorio funciona correctamente jugando una partida contra sí mismo. Utiliza la función `juga_joc` de la clase `Joc`.

In [34]:
joc = TresEnRatlla()
joc.juga_joc(jugador_aleatori, jugador_aleatori)

. . . 
. . X 
. . . 
. O . 
. . X 
. . . 
X O . 
. . X 
. . . 
X O . 
. . X 
O . . 
X O . 
. X X 
O . . 
X O O 
. X X 
O . . 
X O O 
X X X 
O . . 
X O O 
X X X 
O . . 


1

### Función `minimax`

5. Implementa la función `minimax`, que recibe un juego y un estado y devuelve un movimiento teniendo en cuenta la utilidad de los estados resultantes de los movimientos posibles.

In [35]:
def minimax(joc, estat):
    def maximitza(joc, estat):
        if joc.test_terminal(estat):
            return joc.utilitat(estat, joc.torn(joc.inicial)), None
        v = -float('inf')
        moviment = None
        for accio in joc.accions(estat):
            v2, _ = minimitza(joc, joc.resultat(estat, accio))
            if v2 > v:
                v = v2
                moviment = accio
        return v, moviment

    def minimitza(joc, estat):
        if joc.test_terminal(estat):
            return joc.utilitat(estat, joc.torn(joc.inicial)), None
        v = float('inf')
        moviment = None
        for accio in joc.accions(estat):
            v2, _ = maximitza(joc, joc.resultat(estat, accio))
            if v2 < v:
                v = v2
                moviment = accio
        return v, moviment

    return maximitza(joc, estat)[1]

6. Comprueba que el jugador `minimax` funciona correctamente jugando contra el jugador aleatorio. `minimax` debería ganar o empatar siempre.

In [36]:
joc = TresEnRatlla()
joc.juga_joc(minimax, jugador_aleatori)

X . . 
. . . 
. . . 
X . . 
. . O 
. . . 
X . X 
. . O 
. . . 
X . X 
. . O 
. O . 
X X X 
. . O 
. O . 
X X X 
. . O 
. O . 


1

### Función `minimax_alphabeta`

7. El coste computacional de `minimax` es muy alto. Implementa `minimax_alphabeta`, que optimiza minimax mediante poda alfa-beta. Comprueba su funcionamiento jugando contra el jugador aleatorio: debería ganar o empatar siempre.

In [37]:
def minimax_alphabeta(joc, estat):
    def maximitza(joc, estat, alfa, beta):
        if joc.test_terminal(estat):
            return joc.utilitat(estat, joc.torn(joc.inicial)), None
        v = -float('inf')
        moviment = None
        for accio in joc.accions(estat):
            v2, _ = minimitza(joc, joc.resultat(estat, accio), alfa, beta)
            if v2 > v:
                v = v2
                moviment = accio
            if v >= beta:
                return v, moviment
            alfa = max(alfa, v)
        return v, moviment

    def minimitza(joc, estat, alfa, beta):
        if joc.test_terminal(estat):
            return joc.utilitat(estat, joc.torn(joc.inicial)), None
        v = float('inf')
        moviment = None
        for accio in joc.accions(estat):
            v2, _ = maximitza(joc, joc.resultat(estat, accio), alfa, beta)
            if v2 < v:
                v = v2
                moviment = accio
            if v <= alfa:
                return v, moviment
            beta = min(beta, v)
        return v, moviment

    return maximitza(joc, estat, -float('inf'), float('inf'))[1]


joc = TresEnRatlla()
joc.juga_joc(minimax_alphabeta, jugador_aleatori)

X . . 
. . . 
. . . 
X . O 
. . . 
. . . 
X . O 
X . . 
. . . 
X . O 
X . . 
. O . 
X . O 
X X . 
. O . 
X . O 
X X O 
. O . 
X . O 
X X O 
X O . 
X . O 
X X O 
X O . 


1

## Cuatro en raya

El cuatro en raya es parecido al tres en raya, pero se juega en un tablero de 7 × 6 y hay que alinear cuatro fichas para ganar. Los movimientos consisten en elegir una columna. La ficha cae hasta la posición vacía más baja de esa columna.

8. Modifica la clase `TresEnRatlla` para implementar el cuatro en raya. Comprueba su funcionamiento con `juga_joc` de la clase `Joc` y dos jugadores aleatorios.

In [38]:
class QuatreEnRatlla(TresEnRatlla):
    """Un joc de tipus Tres en ratlla en el qual només pots fer un moviment a la fila inferior,
    o en una casella directament per sobre d'una casella ocupada.  Tradicionalment
    es juga en un tauler de 7x6 i es requereixen 4 en línia."""

    def __init__(self, h=7, v=6, k=4):
        TresEnRatlla.__init__(self, h, v, k)

    def accions(self, state):
        return [(x, y) for (x, y) in state.moviments
                if y == 1 or (x, y - 1) in state.tauler]

    def display(self, estat):
        """Mostra l'estat actual del joc."""
        tauler = estat.tauler
        for y in range(self.v, 0, -1):
            for x in range(1, self.h + 1):
                print(tauler.get((x, y), '.'), end=' ')
            print()

In [39]:
joc = QuatreEnRatlla()
joc.juga_joc(jugador_aleatori, jugador_aleatori)

. . . . . . . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
X . . . . . . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
X . O . . . . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
X . . . . . . 
X . O . . . . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
X . . . . . . 
X . O O . . . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
X . . . . . . 
X . O O . X . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
X . . . . . . 
X . O O O X . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
X . . . . X . 
X . O O O X . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
. . . . . O . 
X . . . . X . 
X . O O O X . 
. . . . . . . 
. . . . . . . 
. . . . . X . 
. . . . . O . 
X . . . . X . 
X . O O O X . 
. . . . . . . 
. . . . . . . 
. . . . . X . 
. . . . . O . 
X . . . . X . 
X . O O O X O 
. . . . . . . 
. . . . . . . 
. . . . . X . 
. . . . . O . 
X . . X . X . 
X . O O O X O 
. . . . . 

-1

### Minimax con profundidad limitada y evaluación heurística

9. Implementa `minimax_profunditat_limitada`, que recibe un juego, un estado y una profundidad y devuelve el mejor movimiento dentro de ese límite. Si el estado es terminal o se alcanza el límite de profundidad, debe evaluarse mediante `avaluacio`.

Implementa también `avaluacio`, que recibe un juego y un estado y devuelve un valor numérico que indica la calidad del estado para el jugador al que le toca. Puedes utilizar la función que se muestra a continuación o crear otra.

In [40]:
def avaluacio(joc, estat):
    """Retorna un valor numèric que indica la qualitat de l'estat per al jugador que té el torn."""
    def num_k_en_ralla(tauler, jugador, delta_x_y, k):
        """Retorna el nombre de línies de k fitxes en ratlla del jugador en el tauler.
        """
        (delta_x, delta_y) = delta_x_y
        n = 0  # n és el nombre de moviments en línia
        for x in range(1, joc.h + 1):
            for y in range(1, joc.v + 1):
                if tauler.get((x, y)) == jugador:
                    # Comptar les fitxes en línia des d'ací
                    n += 1
                    for m in range(1, k):
                        if tauler.get((x + m * delta_x, y + m * delta_y)) != jugador:
                            n -= 1
                            break
                    if n == k:
                        return 1
                n = 0
        return 0
    
    num = 0
    for i in range(1, joc.k + 1):
        aux = 0
        aux += num_k_en_ralla(estat.tauler, estat.torn, (0, 1), i)
        aux += num_k_en_ralla(estat.tauler, estat.torn, (1, 0), i)
        aux += num_k_en_ralla(estat.tauler, estat.torn, (1, -1), i)
        aux += num_k_en_ralla(estat.tauler, estat.torn, (1, 1), i)
        num += aux * 10 ** i
    return num

In [41]:
def minimax_ab_profunditat_limitada(joc, estat, profunditat=6):
    def maximitza(joc, estat, alfa, beta, profunditat):
        if joc.test_terminal(estat) or profunditat == 0:
            return avaluacio(joc, estat), None
        v = -float('inf')
        moviment = None
        for accio in joc.accions(estat):
            v2, _ = minimitza(joc, joc.resultat(estat, accio), alfa, beta, profunditat - 1)
            if v2 > v:
                v = v2
                moviment = accio
            if v >= beta:
                return v, moviment
            alfa = max(alfa, v)
        return v, moviment

    def minimitza(joc, estat, alfa, beta, profunditat):
        if joc.test_terminal(estat) or profunditat == 0:
            return avaluacio(joc, estat), None
        v = float('inf')
        moviment = None
        for accio in joc.accions(estat):
            v2, _ = maximitza(joc, joc.resultat(estat, accio), alfa, beta, profunditat - 1)
            if v2 < v:
                v = v2
                moviment = accio
            if v <= alfa:
                return v, moviment
            beta = min(beta, v)
        return v, moviment

    return maximitza(joc, estat, -float('inf'), float('inf'), profunditat)[1]

In [42]:
joc = QuatreEnRatlla()
joc.juga_joc(minimax_ab_profunditat_limitada, jugador_aleatori)

. . . . . . . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
X . . . . . . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
O . . . . . . 
X . . . . . . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
X . . . . . . 
O . . . . . . 
X . . . . . . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
X . . . . . . 
O . . . . . . 
X . O . . . . 
. . . . . . . 
. . . . . . . 
X . . . . . . 
X . . . . . . 
O . . . . . . 
X . O . . . . 
. . . . . . . 
. . . . . . . 
X . . . . . . 
X . . . . . . 
O . . . . . . 
X . O . O . . 
. . . . . . . 
X . . . . . . 
X . . . . . . 
X . . . . . . 
O . . . . . . 
X . O . O . . 
. . . . . . . 
X . . . . . . 
X . . . . . . 
X . . . . . . 
O . . . . . . 
X . O O O . . 
X . . . . . . 
X . . . . . . 
X . . . . . . 
X . . . . . . 
O . . . . . . 
X . O O O . . 
X . . . . . . 
X . . . . . . 
X . . . . . . 
X . . . . . . 
O . . . . . . 
X . O O O . . 


1

### Jugador humano

10. Implementa `jugador_huma`, que recibe un juego y un estado y devuelve el movimiento elegido por el usuario. Si el movimiento no es válido, debe volver a pedir uno hasta que lo sea.

Intenta jugar una partida contra el jugador que has ido creando.

In [43]:
def jugador_huma(joc, estat):
    moviment = None
    while moviment not in joc.accions(estat):
        columna = eval(input('Columna? '))
        altura = max([y for (x, y) in estat.tauler if x == columna] + [0])
        moviment = (columna, altura + 1)
    return moviment

In [47]:
joc = QuatreEnRatlla()
joc.juga_joc(minimax_ab_profunditat_limitada, jugador_aleatori)

. . . . . . . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
X . . . . . . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
X . . . . O . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
X . . . . . . 
X . . . . O . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
X . . . . . . 
X . . . O O . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
X . . . . . . 
X . . . . . . 
X . . . O O . 
. . . . . . . 
. . . . . . . 
. . . . . . . 
X . . . . . . 
X . . . O . . 
X . . . O O . 
. . . . . . . 
. . . . . . . 
X . . . . . . 
X . . . . . . 
X . . . O . . 
X . . . O O . 
. . . . . . . 
. . . . . . . 
X . . . . . . 
X . . . . . . 
X . . . O . . 
X . . . O O . 


1